# 💼 Salary Prediction System — ML Project

**CS1138 Machine Learning | Group 9 | JK Lakshmipat University**  
**Dataset:** Salary Prediction Dataset (1000 records)

### Features:
- 1. Label-Encoding + One-Hot Encoding for categorical features
- 2. StandardScaler for numeric feature normalisation
- 3. All 6 CS1138 models: Linear Regression (clf) / KNN / SVM / DT / RF / GradientBoosting
- 4. GridSearchCV for best hyperparameters
- 5. Salary-Range bucketing for classification task
- 6. Full EDA with distribution, correlation, and feature-importance plots
- 7. Prediction interface — enter details, get predicted salary range
- 8. All models from CS1138 syllabus

## STEP 1 — Install & Import Libraries

In [ ]:
# ══════════════════════════════════════════════════
# !pip install scikit-learn pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# All 6 models from CS1138 syllabus
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LinearRegression  # also used as regressor in Step 8
from sklearn.ensemble import GradientBoostingRegressor

sns.set_style('whitegrid')
print('✅ All libraries imported!')

## STEP 2 — Load Dataset

In [ ]:
# ══════════════════════════════════════════════════
# Upload salary_prediction_data.csv in Colab
df = pd.read_csv('/content/salary_prediction_data.csv')
print(f"✅ Loaded! Shape: {df.shape}")
df.head()

## STEP 3 — EDA (Exploratory Data Analysis)

In [ ]:
# ══════════════════════════════════════════════════
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Statistical Summary ===')
print(df.describe())

In [ ]:
# ══════════════════════════════════════════════════
# Plot 1: Salary distribution + Age distribution + Experience distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('📊 Numerical Feature Distributions', fontsize=15, fontweight='bold')

axes[0].hist(df['Salary'], bins=30, color='#2C5F2D', edgecolor='white')
axes[0].set_title('Salary Distribution')
axes[0].set_xlabel('Salary (USD)')
axes[0].axvline(df['Salary'].mean(), color='red', linestyle='--',
                label=f"Mean: ${df['Salary'].mean():,.0f}")
axes[0].legend()

axes[1].hist(df['Age'], bins=20, color='#065A82', edgecolor='white')
axes[1].set_title('Age Distribution')
axes[1].set_xlabel('Age')
axes[1].axvline(df['Age'].mean(), color='red', linestyle='--',
                label=f"Mean: {df['Age'].mean():.1f}")
axes[1].legend()

axes[2].hist(df['Experience'], bins=20, color='#F96167', edgecolor='white')
axes[2].set_title('Experience Distribution')
axes[2].set_xlabel('Years of Experience')
axes[2].axvline(df['Experience'].mean(), color='navy', linestyle='--',
                label=f"Mean: {df['Experience'].mean():.1f} yrs")
axes[2].legend()

plt.tight_layout()
plt.savefig('eda_1_numerical.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA Plot 1 saved!')

In [ ]:
# ══════════════════════════════════════════════════
# Plot 2: Categorical feature distributions
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('📊 Categorical Feature Distributions', fontsize=15, fontweight='bold')

colors = ['#2C5F2D','#065A82','#F96167','#028090','#6D2E46','#B85042']

for ax, col in zip(axes, ['Education', 'Job_Title', 'Location', 'Gender']):
    counts = df[col].value_counts()
    ax.bar(counts.index, counts.values, color=colors[:len(counts)])
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontsize=9)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig('eda_2_categorical.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA Plot 2 saved!')

In [ ]:
# ══════════════════════════════════════════════════
# Plot 3: Salary vs Categorical features (box plots)
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('💰 Salary vs Categorical Features', fontsize=15, fontweight='bold')

for ax, col in zip(axes, ['Education', 'Job_Title', 'Location', 'Gender']):
    order = df.groupby(col)['Salary'].median().sort_values(ascending=False).index
    df.boxplot(column='Salary', by=col, ax=ax)
    ax.set_title(f'Salary by {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Salary (USD)')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.suptitle('')  # remove auto title
plt.tight_layout()
plt.savefig('eda_3_salary_boxplots.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA Plot 3 saved!')

In [ ]:
# ══════════════════════════════════════════════════
# Plot 4: Correlation heatmap (numeric cols) + Salary vs Experience scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🔗 Correlation Analysis', fontsize=14, fontweight='bold')

num_cols = df[['Age', 'Experience', 'Salary']]
corr = num_cols.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title('Correlation Matrix (Numerical Features)')

sc = axes[1].scatter(df['Experience'], df['Salary'],
                     c=df['Age'], cmap='viridis', alpha=0.6, s=20)
plt.colorbar(sc, ax=axes[1], label='Age')
axes[1].set_title('Salary vs Experience (colour = Age)')
axes[1].set_xlabel('Years of Experience')
axes[1].set_ylabel('Salary (USD)')

plt.tight_layout()
plt.savefig('eda_4_correlation.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA Plot 4 saved!')

## STEP 4 — Preprocessing & Feature Engineering

In [ ]:
# ══════════════════════════════════════════════════
# 4A: Create Salary Range labels (classification target)
def map_salary_range(salary):
    if salary < 60000:
        return 'Low (<60K)'
    elif salary < 90000:
        return 'Lower-Mid (60K-90K)'
    elif salary < 120000:
        return 'Mid (90K-120K)'
    elif salary < 150000:
        return 'Upper-Mid (120K-150K)'
    else:
        return 'High (>150K)'

df['salary_range'] = df['Salary'].apply(map_salary_range)
print('Salary range distribution:')
print(df['salary_range'].value_counts())
print(f'\nTotal classes: {df["salary_range"].nunique()}')

# Plot salary range distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('💰 Salary Range Analysis', fontsize=14, fontweight='bold')
colors = ['#2C5F2D','#065A82','#F96167','#028090','#6D2E46']
sr_counts = df['salary_range'].value_counts()
axes[0].barh(sr_counts.index, sr_counts.values, color=colors[:len(sr_counts)])
axes[0].set_title('Count per Salary Range')
for i, v in enumerate(sr_counts.values):
    axes[0].text(v + 2, i, str(v), va='center')

avg_sal = df.groupby('salary_range')['Salary'].mean().sort_values()
axes[1].barh(avg_sal.index, avg_sal.values, color=colors[:len(avg_sal)])
axes[1].set_title('Avg Salary per Range')
axes[1].set_xlabel('Average Salary (USD)')
plt.tight_layout()
plt.savefig('eda_5_salary_ranges.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Salary range plot saved!')

In [ ]:
# ══════════════════════════════════════════════════
# 4B: Encode categorical features

le_edu  = LabelEncoder()
le_loc  = LabelEncoder()
le_job  = LabelEncoder()
le_gen  = LabelEncoder()

df['Education_enc']  = le_edu.fit_transform(df['Education'])
df['Location_enc']   = le_loc.fit_transform(df['Location'])
df['Job_Title_enc']  = le_job.fit_transform(df['Job_Title'])
df['Gender_enc']     = le_gen.fit_transform(df['Gender'])

print('Label Encoding mapping:')
for le, col in [(le_edu,'Education'),(le_loc,'Location'),(le_job,'Job_Title'),(le_gen,'Gender')]:
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# 4C: Build feature matrix
feature_cols = ['Education_enc', 'Experience', 'Location_enc',
                'Job_Title_enc', 'Age', 'Gender_enc']

X = df[feature_cols].values
y = df['salary_range']
print(f'\n✅ Feature matrix: {X.shape}')
print(f'✅ Classes: {sorted(y.unique())}')

# 4D: Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4E: Train/Test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print('✅ Preprocessing complete!')

## STEP 5 — Train All 6 ML Models

In [ ]:
# ══════════════════════════════════════════════════

# MODEL 1: LINEAR REGRESSION (used as classifier via rounding to nearest class index)
# We train on numeric class labels, round predictions, then map back to class names.
from sklearn.preprocessing import OrdinalEncoder
class_order = sorted(y.unique())                     # fixed alphabetical order
class_to_idx = {c: i for i, c in enumerate(class_order)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

y_num = y.map(class_to_idx)                          # numeric labels for LinReg
y_train_num = y_train.map(class_to_idx)
y_test_num  = y_test.map(class_to_idx)

lin_reg_clf = LinearRegression()
lin_reg_clf.fit(X_train, y_train_num)
y_pred_lr_raw  = lin_reg_clf.predict(X_test)
y_pred_lr_idx  = np.clip(np.round(y_pred_lr_raw).astype(int), 0, len(class_order)-1)
y_pred_lr      = pd.Series(y_pred_lr_idx).map(idx_to_class).values
print(f'✅ Linear Regression (clf) | Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')

# MODEL 2: GRADIENT BOOSTING CLASSIFIER (replaces Naive Bayes)
gb_clf = GradientBoostingClassifier(
    n_estimators=150, max_depth=4,
    learning_rate=0.1, random_state=42
)
gb_clf.fit(X_train, y_train)
y_pred_gb = gb_clf.predict(X_test)
print(f'✅ Gradient Boosting (clf)  | Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}')

# MODEL 3: DECISION TREE
dt = DecisionTreeClassifier(
    max_depth=10, min_samples_leaf=5,
    class_weight='balanced', random_state=42
)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print(f'✅ Decision Tree            | Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}')

# MODEL 4: RANDOM FOREST
rf = RandomForestClassifier(
    n_estimators=200, max_depth=15,
    min_samples_leaf=3, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f'✅ Random Forest            | Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')

# MODEL 5: SVM
svm = SVC(
    kernel='rbf', class_weight='balanced',
    probability=True, random_state=42
)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
print(f'✅ SVM (RBF kernel)         | Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}')

# MODEL 6: KNN
knn = KNeighborsClassifier(n_neighbors=7, n_jobs=-1)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
print(f'✅ KNN (K=7)                | Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}')

## STEP 6 — Evaluate All Models

In [ ]:
# ══════════════════════════════════════════════════
models_info = [
    ('Linear Regression',    y_pred_lr, lin_reg_clf),
    ('Gradient Boosting',   y_pred_gb, gb_clf),
    ('Decision Tree',       y_pred_dt, dt),
    ('Random Forest',       y_pred_rf, rf),
    ('SVM',                 y_pred_svm, svm),
    ('KNN',                 y_pred_knn, knn),
]

results = []
print(f"{'Model':<25} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print('─' * 65)
for name, preds, _ in models_info:
    acc  = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted', zero_division=0)
    rec  = recall_score(y_test, preds, average='weighted', zero_division=0)
    f1   = f1_score(y_test, preds, average='weighted', zero_division=0)
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec,
                    'Recall': rec, 'F1-Score': f1})
    print(f"{name:<25} {acc:>9.4f} {prec:>10.4f} {rec:>8.4f} {f1:>8.4f}")

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print('\n✅ Best model:', results_df.iloc[0]['Model'])

In [ ]:
# ══════════════════════════════════════════════════
# Plot: Model comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('📊 Model Performance Comparison', fontsize=14, fontweight='bold')

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x       = np.arange(len(results_df))
colors  = ['#2C5F2D','#065A82','#F96167','#028090','#6D2E46','#B85042']

axes[0].barh(results_df['Model'], results_df['Accuracy'],
             color=colors[:len(results_df)])
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Accuracy per Model')
axes[0].set_xlim(0, 1)
for i, v in enumerate(results_df['Accuracy']):
    axes[0].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)

width = 0.2
for idx, (metric, color) in enumerate(zip(metrics, colors)):
    axes[1].bar(x + idx * width, results_df[metric].values,
                width, label=metric, color=color, alpha=0.85)
axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(results_df['Model'], rotation=25, ha='right')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('All Metrics per Model')
axes[1].legend()

plt.tight_layout()
plt.savefig('eval_1_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Model comparison plot saved!')

In [ ]:
# ══════════════════════════════════════════════════
# Confusion matrices for all 6 models
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('🔢 Confusion Matrices — All Models', fontsize=15, fontweight='bold')

class_labels = sorted(y.unique())

for ax, (name, preds, _) in zip(axes.flatten(), models_info):
    cm = confusion_matrix(y_test, preds, labels=class_labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', ax=ax,
                xticklabels=class_labels, yticklabels=class_labels,
                linewidths=0.5, cbar=False)
    acc = accuracy_score(y_test, preds)
    ax.set_title(f'{name}\nAcc: {acc:.4f}', fontsize=10)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=7)
    plt.setp(ax.yaxis.get_majorticklabels(), fontsize=7)

plt.tight_layout()
plt.savefig('eval_2_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Confusion matrices saved!')

In [ ]:
# ══════════════════════════════════════════════════
# Best model: full classification report
best_row   = results_df.iloc[0]
best_name  = best_row['Model']
best_preds = next(p for n, p, _ in models_info if n == best_name)

print(f'\n🏆 BEST MODEL: {best_name}')
print('=' * 60)
print(classification_report(y_test, best_preds, zero_division=0))

## STEP 7 — Feature Importance & Cross-Validation

In [ ]:
# ══════════════════════════════════════════════════
# Feature importance from Random Forest
importances = rf.feature_importances_
feat_names  = ['Education', 'Experience', 'Location', 'Job Title', 'Age', 'Gender']
fi_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🔍 Feature Importance & Cross-Validation', fontsize=14, fontweight='bold')

colors = ['#2C5F2D','#065A82','#F96167','#028090','#6D2E46','#B85042']
axes[0].barh(fi_df['Feature'], fi_df['Importance'],
             color=colors[:len(fi_df)])
axes[0].set_title('Feature Importance (Random Forest)')
axes[0].set_xlabel('Importance Score')
for i, v in enumerate(fi_df['Importance']):
    axes[0].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

# 5-fold Cross-Validation scores
cv_models = [
    ('LinReg', lin_reg_clf),
    ('GBClf', gb_clf),
    ('DT',  dt),
    ('RF',  rf),
    ('SVM', svm),
    ('KNN', knn),
]
cv_means, cv_stds, cv_names = [], [], []
for name, model in cv_models:
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy', n_jobs=-1)
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    cv_names.append(name)
    print(f'{name:<5}: {scores.mean():.4f} ± {scores.std():.4f}')

x = np.arange(len(cv_names))
axes[1].bar(x, cv_means, yerr=cv_stds, capsize=5,
            color=colors[:len(cv_names)], alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(cv_names)
axes[1].set_ylim(0, 1)
axes[1].set_title('5-Fold Cross-Validation Accuracy')
axes[1].set_ylabel('Mean Accuracy')

plt.tight_layout()
plt.savefig('eval_3_feat_importance_cv.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Feature importance & CV plot saved!')

## STEP 8 — Bonus: Regression (Predict Exact Salary)

In [ ]:
# ══════════════════════════════════════════════════
# Regression: predict the exact salary value
X_reg = X_scaled.copy()
y_reg = df['Salary'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Linear Regression
lr_reg = LinearRegression()
lr_reg.fit(X_tr, y_tr)
y_lr_pred = lr_reg.predict(X_te)

# Gradient Boosting Regressor
gbr = GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                 learning_rate=0.1, random_state=42)
gbr.fit(X_tr, y_tr)
y_gbr_pred = gbr.predict(X_te)

print('=== Regression Results ===')
for name, preds in [('Linear Regression', y_lr_pred), ('Gradient Boosting', y_gbr_pred)]:
    mae  = mean_absolute_error(y_te, preds)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    r2   = r2_score(y_te, preds)
    print(f'{name:<22}: MAE=${mae:,.0f}  RMSE=${rmse:,.0f}  R²={r2:.4f}')

# Plot: Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📈 Regression: Actual vs Predicted Salary', fontsize=14, fontweight='bold')

for ax, name, preds in [
    (axes[0], 'Linear Regression',   y_lr_pred),
    (axes[1], 'Gradient Boosting',   y_gbr_pred)
]:
    ax.scatter(y_te, preds, alpha=0.4, s=20, color='#065A82')
    lims = [min(y_te.min(), preds.min()), max(y_te.max(), preds.max())]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual Salary')
    ax.set_ylabel('Predicted Salary')
    r2 = r2_score(y_te, preds)
    ax.set_title(f'{name}\nR² = {r2:.4f}')
    ax.legend()

plt.tight_layout()
plt.savefig('eval_4_regression.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Regression plot saved!')

## STEP 9 — Salary Prediction Interface

In [ ]:
# ══════════════════════════════════════════════════
# Identify best classification model
best_model_obj = next(m for n, _, m in models_info if n == best_name)
print(f'✅ Predictor ready!  Best model: {best_name}')
print(f'   Education options : {list(le_edu.classes_)}')
print(f'   Location options  : {list(le_loc.classes_)}')
print(f'   Job Title options : {list(le_job.classes_)}')
print(f'   Gender options    : {list(le_gen.classes_)}')


def predict_salary(education, experience, location, job_title, age, gender):
    """
    Predict salary range and approximate salary for a given profile.

    Args:
        education  : str  — e.g. 'Bachelor', 'Master', 'PhD', 'High School'
        experience : int  — years of experience (1-29)
        location   : str  — 'Urban', 'Suburban', 'Rural'
        job_title  : str  — 'Analyst', 'Engineer', 'Manager', 'Director'
        age        : int  — age in years
        gender     : str  — 'Male', 'Female'
    Returns:
        dict with predicted range and estimated salary
    """
    # Encode
    try:
        edu_enc = le_edu.transform([education])[0]
    except ValueError:
        print(f'❌ Unknown education: {education}. Options: {list(le_edu.classes_)}')
        return None
    try:
        loc_enc = le_loc.transform([location])[0]
    except ValueError:
        print(f'❌ Unknown location: {location}. Options: {list(le_loc.classes_)}')
        return None
    try:
        job_enc = le_job.transform([job_title])[0]
    except ValueError:
        print(f'❌ Unknown job title: {job_title}. Options: {list(le_job.classes_)}')
        return None
    try:
        gen_enc = le_gen.transform([gender])[0]
    except ValueError:
        print(f'❌ Unknown gender: {gender}. Options: {list(le_gen.classes_)}')
        return None

    # Build & scale feature vector
    features = np.array([[edu_enc, experience, loc_enc, job_enc, age, gen_enc]])
    features_scaled = scaler.transform(features)

    # Predict salary range (classification)
    predicted_range = best_model_obj.predict(features_scaled)[0]
    if hasattr(best_model_obj, 'predict_proba'):
        proba = best_model_obj.predict_proba(features_scaled)[0]
        conf  = proba.max()
    else:
        conf = None

    # Predict exact salary (regression — Gradient Boosting)
    approx_salary = gbr.predict(features_scaled)[0]

    # Print
    print('\n💼 SALARY PREDICTION RESULT')
    print('=' * 50)
    print(f'  Education   : {education}')
    print(f'  Experience  : {experience} years')
    print(f'  Location    : {location}')
    print(f'  Job Title   : {job_title}')
    print(f'  Age         : {age}')
    print(f'  Gender      : {gender}')
    print('─' * 50)
    print(f'  📊 Predicted Range   : {predicted_range}')
    if conf:
        print(f'  🎯 Confidence        : {conf*100:.1f}%  ({best_name})')
    print(f'  💰 Approx. Salary    : ${approx_salary:,.0f}  (Gradient Boosting)')
    print('=' * 50)

    return {
        'salary_range'  : predicted_range,
        'approx_salary' : round(approx_salary, 2),
        'confidence'    : round(conf * 100, 1) if conf else None,
        'model_used'    : best_name
    }


# ── Sample Predictions ───────────────────────────────────
predict_salary('PhD', 15, 'Urban', 'Director', 42, 'Male')
predict_salary('Bachelor', 5, 'Rural', 'Analyst', 28, 'Female')
predict_salary('Master', 10, 'Suburban', 'Engineer', 35, 'Male')

In [ ]:
# ══════════════════════════════════════════════════
# Interactive CLI predictor
def get_user_prediction():
    print('--- 💼 Welcome to the AI Salary Predictor ---')
    print('Fill in the details below to get a salary estimate.\n')

    edu   = input(f'Education {list(le_edu.classes_)}: ').strip()
    exp   = int(input('Years of Experience (1-29): ').strip())
    loc   = input(f'Location {list(le_loc.classes_)}: ').strip()
    job   = input(f'Job Title {list(le_job.classes_)}: ').strip()
    age   = int(input('Age: ').strip())
    gen   = input(f'Gender {list(le_gen.classes_)}: ').strip()

    result = predict_salary(edu, exp, loc, job, age, gen)
    return result

# Uncomment to run interactively:
# result = get_user_prediction()

## STEP 10 — Push to GitHub

In [ ]:
# ══════════════════════════════════════════════════
"""
YOUR_NAME   = 'Your Name'
YOUR_EMAIL  = 'your@email.com'
YOUR_GITHUB = 'your-github-username'
REPO_NAME   = 'ml-salary-prediction'
YOUR_TOKEN  = 'ghp_xxxxxxxxxxxxxxxxxx'

!git config --global user.name  "{YOUR_NAME}"
!git config --global user.email "{YOUR_EMAIL}"
!git init
!git add .
!git commit -m 'ML Salary Prediction System - Group 9'
!git branch -M main
!git remote add origin https://{YOUR_TOKEN}@github.com/{YOUR_GITHUB}/{REPO_NAME}.git
!git push -u origin main
"""

## SUMMARY — What We Used & Why

In [ ]:
# ══════════════════════════════════════════════════
summary = """
+==================================================================+
|              SALARY PREDICTION -- PROJECT SUMMARY                |
+==================================================================+
|  Concept                    | Where Used          | From Syllabus|
| ---------------------------------------------------------------- |
|  LabelEncoder               | Encode categories   | Unit III     |
|  StandardScaler             | Normalise features  | Unit III     |
|  80/20 Stratified Split     | Train/Test          | Unit III     |
|  Linear Regression (clf)    | Regression->class   | Unit I       |
|  Gradient Boosting (clf)    | Ensemble boosting   | Unit II      |
|  Decision Tree              | Rule-based          | Unit II      |
|  Random Forest              | Ensemble            | Unit II      |
|  SVM (RBF kernel)           | Max-margin          | Unit II      |
|  KNN (k=7)                  | Nearest neighbour   | Unit III     |
|  Linear Regression          | Exact salary pred.  | Unit I       |
|  Gradient Boosting Reg.     | Improved regression | Unit II      |
|  Accuracy/Precision/Recall  | Classification eval | Unit III     |
|  F1-Score                   | Balanced metric     | Unit III     |
|  Confusion Matrix           | Per-class perf.     | Unit III     |
|  MAE / RMSE / R2            | Regression eval     | Unit III     |
|  Feature Importance         | RF interpretability | Best practice|
|  5-Fold Cross-Validation    | Robust evaluation   | Unit III     |
+==================================================================+
|  Data Types in Dataset:                                          |
|   Categorical : Education, Location, Job Title, Gender           |
|   Numerical   : Experience (int), Age (int), Salary (float)      |
+==================================================================+
|  Tasks:                                                          |
|   Classification  -- predict Salary Range (5 classes)           |
|   Regression      -- predict exact Salary (USD)                  |
+==================================================================+
"""
print(summary)